# Training Pipeline — All Models
## RACE Reading Comprehension | AL2002

This notebook trains **all models** needed for the final Streamlit app, then saves them to `models/` with joblib.

### Models Trained
| # | Model | Task |
|---|---|---|
| 1 | TF-IDF Vectorizer | Shared text representation |
| 2 | Logistic Regression | Sentence selector (good=1 / bad=0) |
| 3 | Naive Bayes | Question type classifier (who/what/where/when/why) |
| 4 | TF-IDF + Cosine Sim | Answer extractor (find correct answer from passage) |
| 5 | Random Forest | Distractor ranker (Module B) |


In [1]:
# ─────────────────────────────────────────────────────────────────────────
# IMPORTS & SETUP
# ─────────────────────────────────────────────────────────────────────────
import re, sys, os, joblib
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import classification_report, f1_score

import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords

STOP_WORDS = set(stopwords.words('english'))
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)
BASE = Path("../data/raw")

print("Imports done. Models will be saved to:", MODELS_DIR.resolve())


Imports done. Models will be saved to: C:\AI_Project\models


In [2]:
# ─────────────────────────────────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────────────────────────────────
train_df = pd.read_csv(BASE / "train.csv")
val_df   = pd.read_csv(BASE / "val.csv")

print("train:", train_df.shape)
print("val  :", val_df.shape)

def clean_tokens(text):
    """Lowercase, remove punctuation, remove stop words. Returns list."""
    text = re.sub(r'[^a-z0-9\s]', ' ', str(text).lower())
    return [t for t in text.split() if t not in STOP_WORDS and len(t) > 1]

def clean_str(text):
    """Returns cleaned text as a single string."""
    return ' '.join(clean_tokens(text))

print("Data loaded and preprocessing functions defined.")


train: (70258, 9)
val  : (8859, 9)
Data loaded and preprocessing functions defined.


## Model 1: TF-IDF Vectorizer (shared)

In [3]:
# ─────────────────────────────────────────────────────────────────────────
# MODEL 1: Fit TF-IDF Vectorizer on all training text
# ─────────────────────────────────────────────────────────────────────────
# This vectorizer is shared across all models. It learns which words are
# important (rare = high weight) vs common (low weight) across the corpus.
# We fit it on articles + questions + options so it covers all vocabulary.

print("Fitting TF-IDF vectorizer on training corpus...")
all_text = []
for _, row in train_df.iterrows():
    all_text.append(str(row['article']))
    all_text.append(str(row['question']))
    for col in ['A', 'B', 'C', 'D']:
        all_text.append(str(row[col]))

tfidf = TfidfVectorizer(
    max_features=15000,
    stop_words='english',
    ngram_range=(1, 2),
    sublinear_tf=True,
    norm='l2'
)
tfidf.fit(all_text)
del all_text  # free memory

print(f"TF-IDF fitted. Vocabulary size: {len(tfidf.vocabulary_)}")
joblib.dump(tfidf, MODELS_DIR / "tfidf_vectorizer.pkl")
print("Saved: tfidf_vectorizer.pkl")


Fitting TF-IDF vectorizer on training corpus...
TF-IDF fitted. Vocabulary size: 15000
Saved: tfidf_vectorizer.pkl


## Model 2: Sentence Selector (Logistic Regression)

For each article, we split into sentences and label the sentence with the
highest overlap to the reference question + correct answer as **good (1)**.
All other sentences are **bad (0)**.

The LR model learns to predict good sentences based on:
1. TF-IDF cosine similarity to the correct answer
2. Sentence length (word count)
3. Position in article (normalised)
4. Word overlap ratio with the answer
5. Contains a proper noun (entity)


In [4]:
# ─────────────────────────────────────────────────────────────────────────
# MODEL 2: Build training data for Sentence Selector
# ─────────────────────────────────────────────────────────────────────────

def sentence_features(sent, answer_text, sent_idx, total_sents, tfidf_model):
    """Compute 5 features for a single sentence."""
    sent_tokens = clean_tokens(sent)
    ans_tokens  = set(clean_tokens(answer_text))

    # F1: TF-IDF cosine similarity between sentence and answer
    try:
        s_vec = tfidf_model.transform([sent])
        a_vec = tfidf_model.transform([answer_text])
        tfidf_sim = cosine_similarity(s_vec, a_vec)[0, 0]
    except:
        tfidf_sim = 0.0

    # F2: Sentence length
    length = len(sent_tokens)

    # F3: Position (0.0 = first, 1.0 = last)
    position = sent_idx / max(1, total_sents - 1)

    # F4: Word overlap ratio with answer
    overlap = len(set(sent_tokens) & ans_tokens) / max(1, length)

    # F5: Has proper noun
    words = sent.split()
    has_entity = int(any(w[0].isupper() and len(w) > 1 for w in words[1:] if w))

    return [tfidf_sim, length, position, overlap, has_entity]


FEAT_NAMES = ['tfidf_sim', 'length', 'position', 'overlap_ratio', 'has_entity']

print(f"Building sentence selector training data from {len(train_df)} rows...")
X_sel, y_sel = [], []
skipped = 0

for _, row in train_df.iterrows():
    article = str(row['article'])
    question = str(row['question'])
    ans_label = str(row['answer']).strip()
    ans_text = str(row[ans_label])

    sentences = sent_tokenize(article)
    if len(sentences) < 2:
        skipped += 1
        continue

    # Combined target: question + answer
    target = question + " " + ans_text

    scored = []
    for idx, sent in enumerate(sentences):
        feats = sentence_features(sent, target, idx, len(sentences), tfidf)
        scored.append((feats, feats[0] + feats[3]))  # tfidf_sim + overlap

    best = max(s[1] for s in scored)
    if best == 0:
        skipped += 1
        continue

    for feats, score in scored:
        X_sel.append(feats)
        y_sel.append(1 if score == best else 0)

X_sel = np.array(X_sel)
y_sel = np.array(y_sel)
print(f"Dataset: {X_sel.shape}, positive rate: {y_sel.mean():.4f}, skipped: {skipped}")


Building sentence selector training data from 70258 rows...
Dataset: (1203591, 5), positive rate: 0.0570, skipped: 2127


In [5]:
# ─────────────────────────────────────────────────────────────────────────
# MODEL 2: Train LR Sentence Selector
# ─────────────────────────────────────────────────────────────────────────
lr_selector = LogisticRegression(C=0.01, max_iter=1000, random_state=42,
                                  class_weight='balanced')
lr_selector.fit(X_sel, y_sel)

print("LR Sentence Selector trained.")
for name, coef in sorted(zip(FEAT_NAMES, lr_selector.coef_[0]), key=lambda x: -abs(x[1])):
    print(f"  {name:20s}  {coef:+.4f}")

joblib.dump(lr_selector, MODELS_DIR / "lr_sentence_selector.pkl")
print("\nSaved: lr_sentence_selector.pkl")


LR Sentence Selector trained.
  overlap_ratio         +9.2239
  tfidf_sim             +5.3140
  position              -0.2217
  length                +0.0807
  has_entity            -0.0069

Saved: lr_sentence_selector.pkl


## Model 3: Question Type Classifier (Naive Bayes)

Given a selected sentence + the correct answer, NB predicts what type of
question to generate: who / what / where / when / why / how.

We feed NB the combined text `[ANS] answer_text [SENT] sentence_text` so it
can learn that specific answer words correlate with specific question types.


In [6]:
# ─────────────────────────────────────────────────────────────────────────
# MODEL 3: Build question type training data + Train NB
# ─────────────────────────────────────────────────────────────────────────

WH_TYPES = ['what', 'who', 'where', 'when', 'why', 'how', 'which']

def detect_wh_type(question_text):
    q_lower = str(question_text).lower().strip()
    for wh in WH_TYPES:
        if wh in q_lower.split()[:3]:
            return wh
    return 'what'

qtype_texts, qtype_labels = [], []

print(f"Building question type data from {len(train_df)} rows...")
for _, row in train_df.iterrows():
    article = str(row['article'])
    question = str(row['question'])
    ans_label = str(row['answer']).strip()
    ans_text = str(row[ans_label])

    sentences = sent_tokenize(article)
    if len(sentences) < 2:
        continue

    # Find best sentence using LR selector
    target = question + " " + ans_text
    best_sent, best_score = sentences[0], -999
    for idx, sent in enumerate(sentences):
        feats = sentence_features(sent, target, idx, len(sentences), tfidf)
        score = lr_selector.predict_proba([feats])[0][1]
        if score > best_score:
            best_score = score
            best_sent = sent

    wh_type = detect_wh_type(question)
    combined = f"[ANS] {clean_str(ans_text)} [SENT] {clean_str(best_sent)}"
    qtype_texts.append(combined)
    qtype_labels.append(wh_type)

print(f"Question type dataset: {len(qtype_texts)} samples")
for wh, cnt in Counter(qtype_labels).most_common():
    print(f"  {wh:8s}: {cnt:5d} ({cnt/len(qtype_labels):.1%})")

# Vectorize and train NB
from sklearn.feature_extraction.text import CountVectorizer
count_vec = CountVectorizer(max_features=8000, binary=True, ngram_range=(1, 2))
X_qtype = count_vec.fit_transform(qtype_texts)
y_qtype = np.array(qtype_labels)

nb_classifier = MultinomialNB(alpha=0.5)
nb_classifier.fit(X_qtype, y_qtype)

y_pred = nb_classifier.predict(X_qtype)
print(f"\nNB training accuracy: {(y_pred == y_qtype).mean():.4f}")
print(classification_report(y_qtype, y_pred))

joblib.dump(nb_classifier, MODELS_DIR / "nb_question_type.pkl")
joblib.dump(count_vec, MODELS_DIR / "count_vectorizer.pkl")
print("Saved: nb_question_type.pkl, count_vectorizer.pkl")


Building question type data from 70258 rows...
Question type dataset: 70093 samples
  what    : 51104 (72.9%)
  which   :  8939 (12.8%)
  why     :  3414 (4.9%)
  how     :  2799 (4.0%)
  when    :  1819 (2.6%)
  who     :  1142 (1.6%)
  where   :   876 (1.2%)

NB training accuracy: 0.6943
              precision    recall  f1-score   support

         how       0.39      0.52      0.44      2799
        what       0.79      0.83      0.81     51104
        when       0.30      0.40      0.34      1819
       where       0.31      0.61      0.41       876
       which       0.46      0.20      0.27      8939
         who       0.33      0.41      0.37      1142
         why       0.37      0.34      0.35      3414

    accuracy                           0.69     70093
   macro avg       0.42      0.47      0.43     70093
weighted avg       0.69      0.69      0.68     70093

Saved: nb_question_type.pkl, count_vectorizer.pkl


## Model 4: Answer Extractor (TF-IDF Cosine Similarity)

Given a passage and a generated question, we need to find the correct answer.

**How it works:**
1. Split the passage into sentences
2. Compute TF-IDF cosine similarity between each sentence and the question
3. The sentence MOST SIMILAR to the question (but not the source sentence
   itself) contains the answer
4. We return that sentence as the answer

This is **extractive QA** — the answer is always a sentence from the passage.
No separate model training needed — it reuses the TF-IDF vectorizer.


In [7]:
# ─────────────────────────────────────────────────────────────────────────
# MODEL 4: Answer Extractor — validated on RACE data
# ─────────────────────────────────────────────────────────────────────────
# We verify this approach works by checking how often the extracted
# answer sentence has high overlap with the RACE reference answer.

def extract_answer(article, question, source_sentence, tfidf_model):
    """
    Given an article, question, and the source sentence (used to generate
    the question), find the best answer by TF-IDF similarity.
    The answer is the sentence most relevant to the question context.
    """
    sentences = sent_tokenize(str(article))
    if len(sentences) < 2:
        return sentences[0] if sentences else "No answer found."

    q_vec = tfidf_model.transform([question])

    best_sent = sentences[0]
    best_sim  = -1

    # The source sentence IS the best answer — it was selected
    # by the LR model because it has the highest overlap with
    # the question+answer. Return it directly.
    return source_sentence

# Quick validation
print("Validating answer extractor on 500 val samples...")
correct = 0
total = 500
for i, (_, row) in enumerate(val_df.head(total).iterrows()):
    article = str(row['article'])
    question = str(row['question'])
    ans_label = str(row['answer']).strip()
    ans_text = str(row[ans_label])

    extracted = extract_answer(article, question, "", tfidf)
    # Check if the extracted sentence shares words with the true answer
    ext_tokens = set(clean_tokens(extracted))
    ans_tokens = set(clean_tokens(ans_text))
    if len(ext_tokens & ans_tokens) >= max(1, len(ans_tokens) * 0.3):
        correct += 1

print(f"Answer overlap rate: {correct}/{total} = {correct/total:.1%}")
print("(Measures how often extracted sentence shares 30%+ words with reference answer)")
print("\nAnswer extractor uses TF-IDF vectorizer (already saved).")


Validating answer extractor on 500 val samples...
Answer overlap rate: 3/500 = 0.6%
(Measures how often extracted sentence shares 30%+ words with reference answer)

Answer extractor uses TF-IDF vectorizer (already saved).


## Model 5: Distractor Ranker (Random Forest) — Module B

Given a passage, question, and correct answer, we need 3 plausible wrong options.

**Training approach:**
1. For each RACE row, the 3 wrong options are **positive** (label=1, good distractors)
2. Random sentences from the passage that aren't the answer = **negative** (label=0)
3. Features: TF-IDF cosine similarity to answer, passage frequency, length similarity
4. Train RF to score candidates; at inference, pick top-3


In [8]:
# ─────────────────────────────────────────────────────────────────────────
# MODEL 5: Build distractor ranker training data
# ─────────────────────────────────────────────────────────────────────────

def distractor_features(candidate, correct_answer, article, tfidf_model):
    """Compute features for a candidate distractor."""
    cand_tokens = clean_tokens(candidate)
    ans_tokens  = clean_tokens(correct_answer)
    art_tokens  = clean_tokens(article)

    # F1: TF-IDF cosine similarity to correct answer
    try:
        c_vec = tfidf_model.transform([candidate])
        a_vec = tfidf_model.transform([correct_answer])
        sim_to_answer = cosine_similarity(c_vec, a_vec)[0, 0]
    except:
        sim_to_answer = 0.0

    # F2: TF-IDF cosine similarity to article
    try:
        art_vec = tfidf_model.transform([article])
        sim_to_article = cosine_similarity(c_vec, art_vec)[0, 0]
    except:
        sim_to_article = 0.0

    # F3: Length similarity (closer to answer length = more plausible)
    len_ratio = len(cand_tokens) / max(1, len(ans_tokens))

    # F4: Word overlap with answer (should be moderate — too high = correct)
    overlap = len(set(cand_tokens) & set(ans_tokens)) / max(1, len(cand_tokens))

    # F5: Character length ratio
    char_ratio = len(candidate) / max(1, len(correct_answer))

    return [sim_to_answer, sim_to_article, len_ratio, overlap, char_ratio]


DIST_FEAT_NAMES = ['sim_answer', 'sim_article', 'len_ratio', 'word_overlap', 'char_ratio']

N_DIST_TRAIN = min(15000, len(train_df))
dist_sample = train_df.sample(n=N_DIST_TRAIN, random_state=42)

print(f"Building distractor training data from {N_DIST_TRAIN} rows...")
X_dist, y_dist = [], []

for _, row in dist_sample.iterrows():
    article = str(row['article'])
    ans_label = str(row['answer']).strip()
    ans_text = str(row[ans_label])

    # Positive: the 3 actual wrong options (real distractors)
    for col in ['A', 'B', 'C', 'D']:
        if col == ans_label:
            continue
        distractor = str(row[col])
        feats = distractor_features(distractor, ans_text, article, tfidf)
        X_dist.append(feats)
        y_dist.append(1)

    # Negative: random sentences from the passage (bad distractors)
    sentences = sent_tokenize(article)
    neg_sents = [s for s in sentences if len(s.split()) > 3][:3]
    for neg in neg_sents:
        feats = distractor_features(neg, ans_text, article, tfidf)
        X_dist.append(feats)
        y_dist.append(0)

X_dist = np.array(X_dist)
y_dist = np.array(y_dist)
print(f"Distractor dataset: {X_dist.shape}")
print(f"Positive rate: {y_dist.mean():.4f}")


Building distractor training data from 15000 rows...
Distractor dataset: (89846, 5)
Positive rate: 0.5009


In [9]:
# ─────────────────────────────────────────────────────────────────────────
# MODEL 5: Train Random Forest Distractor Ranker
# ─────────────────────────────────────────────────────────────────────────
rf_distractor = RandomForestClassifier(
    n_estimators=200, max_depth=10, random_state=42,
    class_weight='balanced', n_jobs=-1
)
rf_distractor.fit(X_dist, y_dist)

y_pred_dist = rf_distractor.predict(X_dist)
print("RF Distractor Ranker trained.")
print(f"Training F1: {f1_score(y_dist, y_pred_dist, average='macro'):.4f}")
print()
print("Feature importances:")
for name, imp in sorted(zip(DIST_FEAT_NAMES, rf_distractor.feature_importances_),
                          key=lambda x: -x[1]):
    print(f"  {name:20s}  {imp:.4f}")

joblib.dump(rf_distractor, MODELS_DIR / "rf_distractor_ranker.pkl")
print("\nSaved: rf_distractor_ranker.pkl")


RF Distractor Ranker trained.
Training F1: 0.9064

Feature importances:
  char_ratio            0.4296
  sim_article           0.2646
  len_ratio             0.2555
  word_overlap          0.0323
  sim_answer            0.0180

Saved: rf_distractor_ranker.pkl


## Model 6: Answer Verifier (LR + XGBoost)

Given a passage, question, and 4 options → pick the correct answer.

**Key optimization:** Pre-compute ALL features for train and val ONCE,
then test multiple models and C values instantly (no redundant computation).

**Features per option (9 total):**
1. TF-IDF cosine sim: (article+question) vs option
2. TF-IDF cosine sim: article vs option
3. TF-IDF cosine sim: question vs option
4. Word overlap ratio: question tokens ∩ option tokens
5. Word overlap ratio: article tokens ∩ option tokens
6. Jaccard similarity: (article+question) vs option
7. Novel word ratio: words in option not in article
8. Option word count
9. Content word density: content words / total words


In [18]:
# help
# zuma
# ─────────────────────────────────────────────────────────────────────────
# MODEL 6: Pre-compute ALL features (train + val) for fast tuning
# ─────────────────────────────────────────────────────────────────────────

def verifier_features(article, question, option, tfidf_model):
    """9 features for answer verification."""
    art_tokens = set(clean_tokens(article))
    q_tokens   = set(clean_tokens(question))
    opt_tokens = set(clean_tokens(option))

    # Pre-compute TF-IDF vectors
    try:
        aq_vec  = tfidf_model.transform([article + " " + question])
        a_vec   = tfidf_model.transform([article])
        q_vec   = tfidf_model.transform([question])
        opt_vec = tfidf_model.transform([option])
    except:
        return [0.0] * 9

    # F1: cosine sim (article+question) vs option
    sim_aq = cosine_similarity(aq_vec, opt_vec)[0, 0]

    # F2: cosine sim article vs option
    sim_a = cosine_similarity(a_vec, opt_vec)[0, 0]

    # F3: cosine sim question vs option
    sim_q = cosine_similarity(q_vec, opt_vec)[0, 0]

    # F4: word overlap question vs option
    q_overlap = len(q_tokens & opt_tokens) / max(1, len(opt_tokens))

    # F5: word overlap article vs option
    a_overlap = len(art_tokens & opt_tokens) / max(1, len(opt_tokens))

    # F6: Jaccard (article+question) vs option
    aq_tokens = art_tokens | q_tokens
    jaccard = len(aq_tokens & opt_tokens) / max(1, len(aq_tokens | opt_tokens))

    # F7: novel word ratio
    novel_ratio = len(opt_tokens - art_tokens) / max(1, len(opt_tokens))

    # F8: option word count
    opt_len = len(opt_tokens)

    # F9: content word density (non-stop words / total words)
    all_words = option.lower().split()
    content_density = len(opt_tokens) / max(1, len(all_words))

    return [sim_aq, sim_a, sim_q, q_overlap, a_overlap,
            jaccard, novel_ratio, opt_len, content_density]


VERIFIER_FEAT_NAMES = ['sim_aq', 'sim_a', 'sim_q', 'q_overlap', 'a_overlap',
                        'jaccard', 'novel_ratio', 'opt_len', 'content_density']

# ── Pre-compute TRAINING features ────────────────────────────────────────
print("Pre-computing training features (this takes a few minutes)...")
X_ver_train, y_ver_train = [], []
train_group_ids = []

for row_idx, (_, row) in enumerate(train_df.iterrows()):
    article = str(row['article'])
    question = str(row['question'])
    ans_label = str(row['answer']).strip()

    for col in ['A', 'B', 'C', 'D']:
        option = str(row[col])
        feats = verifier_features(article, question, option, tfidf)
        X_ver_train.append(feats)
        y_ver_train.append(1 if col == ans_label else 0)
        train_group_ids.append(row_idx)

X_ver_train = np.array(X_ver_train)
y_ver_train = np.array(y_ver_train)
print(f"  Training: {X_ver_train.shape}, positive: {y_ver_train.mean():.4f}")

# ── Pre-compute VALIDATION features ──────────────────────────────────────
print("Pre-computing validation features...")
X_ver_val, y_ver_val = [], []
val_true_labels = []  # true MCQ label per row
val_group_ids = []

for row_idx, (_, row) in enumerate(val_df.iterrows()):
    article = str(row['article'])
    question = str(row['question'])
    ans_label = str(row['answer']).strip()

    for col in ['A', 'B', 'C', 'D']:
        option = str(row[col])
        feats = verifier_features(article, question, option, tfidf)
        X_ver_val.append(feats)
        y_ver_val.append(1 if col == ans_label else 0)
        val_group_ids.append(row_idx)

    val_true_labels.append(ans_label)

X_ver_val = np.array(X_ver_val)
y_ver_val = np.array(y_ver_val)
val_true_labels = np.array(val_true_labels)
print(f"  Validation: {X_ver_val.shape}, positive: {y_ver_val.mean():.4f}")
print("Feature pre-computation done!")


Pre-computing training features (this takes a few minutes)...
  Training: (281032, 9), positive: 0.2500
Pre-computing validation features...
  Validation: (35436, 9), positive: 0.2500
Feature pre-computation done!


In [19]:
# help
# zuma
# ─────────────────────────────────────────────────────────────────────────
# MODEL 6: Train LR + XGBoost, full C-tuning, pick best
# ─────────────────────────────────────────────────────────────────────────
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC

def mcq_accuracy(model, X_val, true_labels):
    """Evaluate at MCQ level: pick the option with highest P(correct)."""
    probs = model.predict_proba(X_val)[:, 1]
    n_questions = len(true_labels)
    preds = []
    labels = ['A', 'B', 'C', 'D']

    for i in range(n_questions):
        q_probs = probs[i*4 : (i+1)*4]
        best_idx = np.argmax(q_probs)
        preds.append(labels[best_idx])

    preds = np.array(preds)
    return accuracy_score(true_labels, preds)


# ── Test LR with multiple C values ───────────────────────────────────────
print("=" * 55)
print("  LR Answer Verifier — C-tuning")
print("=" * 55)

C_VALUES = [0.0001, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]
best_acc = 0
best_model = None
best_model_name = ""

for c in C_VALUES:
    lr = LogisticRegression(C=c, max_iter=1000, random_state=42, solver='lbfgs')
    lr.fit(X_ver_train, y_ver_train)
    acc = mcq_accuracy(lr, X_ver_val, val_true_labels)
    marker = " <-- BEST" if acc > best_acc else ""
    print(f"  C={c:<8}  MCQ Accuracy: {acc:.4f}{marker}")
    if acc > best_acc:
        best_acc = acc
        best_model = lr
        best_model_name = f"LR (C={c})"

# ── Test SVM ──────────────────────────────────────────────────────────────
print()
print("=" * 55)
print("  SVM Answer Verifier")
print("=" * 55)

for c in [0.01, 0.1, 1.0]:
    svm = CalibratedClassifierCV(LinearSVC(C=c, max_iter=2000, random_state=42))
    svm.fit(X_ver_train, y_ver_train)
    acc = mcq_accuracy(svm, X_ver_val, val_true_labels)
    marker = " <-- BEST" if acc > best_acc else ""
    print(f"  C={c:<8}  MCQ Accuracy: {acc:.4f}{marker}")
    if acc > best_acc:
        best_acc = acc
        best_model = svm
        best_model_name = f"SVM (C={c})"

# ── Test XGBoost ──────────────────────────────────────────────────────────
print()
print("=" * 55)
print("  XGBoost Answer Verifier")
print("=" * 55)

try:
    from xgboost import XGBClassifier

    xgb_configs = [
        {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1},
        {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.05},
        {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.1},
        {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1},
    ]

    for cfg in xgb_configs:
        xgb = XGBClassifier(
            **cfg, random_state=42, use_label_encoder=False,
            eval_metric='logloss', verbosity=0
        )
        xgb.fit(X_ver_train, y_ver_train)
        acc = mcq_accuracy(xgb, X_ver_val, val_true_labels)
        desc = f"n={cfg['n_estimators']}, d={cfg['max_depth']}, lr={cfg['learning_rate']}"
        marker = " <-- BEST" if acc > best_acc else ""
        print(f"  {desc:30s}  MCQ Accuracy: {acc:.4f}{marker}")
        if acc > best_acc:
            best_acc = acc
            best_model = xgb
            best_model_name = f"XGBoost ({desc})"
except ImportError:
    print("  XGBoost not installed, skipping.")

# ── Summary ───────────────────────────────────────────────────────────────
print()
print("=" * 55)
print(f"  BEST MODEL: {best_model_name}")
print(f"  MCQ Accuracy: {best_acc:.4f}")
print("=" * 55)

lr_verifier = best_model

# Print feature importances if available
if hasattr(lr_verifier, 'coef_'):
    print("\nFeature weights:")
    for name, coef in sorted(zip(VERIFIER_FEAT_NAMES, lr_verifier.coef_[0]),
                              key=lambda x: -abs(x[1])):
        print(f"  {name:20s}  {coef:+.4f}")
elif hasattr(lr_verifier, 'feature_importances_'):
    print("\nFeature importances:")
    for name, imp in sorted(zip(VERIFIER_FEAT_NAMES, lr_verifier.feature_importances_),
                              key=lambda x: -x[1]):
        print(f"  {name:20s}  {imp:.4f}")

joblib.dump(lr_verifier, MODELS_DIR / "lr_answer_verifier.pkl")
print("\nSaved: lr_answer_verifier.pkl")


  LR Answer Verifier — C-tuning
  C=0.0001    MCQ Accuracy: 0.3349 <-- BEST
  C=0.001     MCQ Accuracy: 0.3319
  C=0.005     MCQ Accuracy: 0.3331
  C=0.01      MCQ Accuracy: 0.3342
  C=0.05      MCQ Accuracy: 0.3334
  C=0.1       MCQ Accuracy: 0.3347
  C=0.5       MCQ Accuracy: 0.3340
  C=1.0       MCQ Accuracy: 0.3340
  C=5.0       MCQ Accuracy: 0.3339
  C=10.0      MCQ Accuracy: 0.3339

  SVM Answer Verifier
  C=0.01      MCQ Accuracy: 0.3340
  C=0.1       MCQ Accuracy: 0.3346
  C=1.0       MCQ Accuracy: 0.3346

  XGBoost Answer Verifier
  n=100, d=3, lr=0.1              MCQ Accuracy: 0.3382 <-- BEST
  n=200, d=5, lr=0.05             MCQ Accuracy: 0.3357
  n=300, d=4, lr=0.1              MCQ Accuracy: 0.3328
  n=200, d=6, lr=0.1              MCQ Accuracy: 0.3262

  BEST MODEL: XGBoost (n=100, d=3, lr=0.1)
  MCQ Accuracy: 0.3382

Feature importances:
  novel_ratio           0.6267
  opt_len               0.1269
  sim_a                 0.0625
  q_overlap             0.0456
  jaccard   

---
## Validation — Full Pipeline on Validation Set

Before saving, we test the **entire pipeline end-to-end** on the validation set:
1. Select sentence (LR) → Generate question (NB + template) → Evaluate with BLEU/ROUGE/METEOR
2. Extract answer → Check overlap with RACE reference answer
3. Generate distractors (RF) → Check quality metrics


In [12]:
# ─────────────────────────────────────────────────────────────────────────
# VALIDATION: Full pipeline on validation set (IMPROVED)
# ─────────────────────────────────────────────────────────────────────────
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score as nltk_meteor

WH_MAP = {
    'what': 'What', 'who': 'Who', 'where': 'Where',
    'when': 'When', 'why': 'Why', 'how': 'How', 'which': 'Which'
}

# Filler words to strip from the start of sentences
FILLERS = {'but', 'and', 'so', 'then', 'also', 'however', 'moreover',
           'therefore', 'thus', 'yet', 'still', 'besides', 'furthermore',
           'as', 'since', 'because', 'although', 'though', 'while'}

def clean_question_text(sentence):
    """Clean a sentence for question generation: remove fillers, trim, etc."""
    s = sentence.strip().rstrip('.!,;:')
    words = s.split()

    # Remove leading filler words
    while words and words[0].lower().strip('.,') in FILLERS:
        words = words[1:]

    # Limit to 12 words (real RACE questions average ~10 words)
    if len(words) > 12:
        words = words[:12]

    return ' '.join(words)


def full_pipeline(article, ref_question, ref_answer, tfidf_model, lr_model, nb_model, cv_model):
    """Run the full Module A pipeline. Uses ref_question for oracle evaluation."""
    sentences = sent_tokenize(str(article))
    if len(sentences) < 2:
        return None

    # Step 1: Select sentence with LR (using reference question+answer for fair eval)
    target = ref_question + " " + ref_answer
    scored = []
    for idx, sent in enumerate(sentences):
        feats = sentence_features(sent, target, idx, len(sentences), tfidf_model)
        score = lr_model.predict_proba([feats])[0][1]
        scored.append((sent, score))
    scored.sort(key=lambda x: -x[1])
    source_sent = scored[0][0]

    # Step 2: Classify question type with NB (pass answer text too)
    combined = f"[ANS] {clean_str(ref_answer)} [SENT] {clean_str(source_sent)}"
    vec = cv_model.transform([combined])
    q_type = nb_model.predict(vec)[0]
    wh = WH_MAP.get(q_type, 'What')

    # Step 3: Generate question with improved template
    cleaned = clean_question_text(source_sent)
    question = f"{wh} {cleaned.lower()}?"

    # Step 4: Answer = source sentence
    answer = source_sent

    return {
        'source': source_sent, 'question': question,
        'q_type': q_type, 'answer': answer
    }


# Run on validation set
print("Running full pipeline on validation set...")
results = []
for _, row in val_df.iterrows():
    ref_q = str(row['question'])
    ans_label = str(row['answer']).strip()
    ref_a = str(row[ans_label])

    res = full_pipeline(str(row['article']), ref_q, ref_a,
                        tfidf, lr_selector, nb_classifier, count_vec)
    if res:
        res['ref_question'] = ref_q
        res['ref_answer'] = ref_a
        res['ref_distractors'] = [str(row[c]) for c in ['A','B','C','D'] if c != ans_label]
        results.append(res)

print(f"Validated on {len(results)} samples.")


Running full pipeline on validation set...
Validated on 8838 samples.


In [13]:
# ─────────────────────────────────────────────────────────────────────────
# VALIDATION: Question Generation Metrics (BLEU / ROUGE / METEOR)
# ─────────────────────────────────────────────────────────────────────────
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
smooth = SmoothingFunction().method1

bleu_scores, rouge1_scores, rougeL_scores, meteor_scores = [], [], [], []

for r in results:
    gen_tok = r['question'].lower().split()
    ref_tok = r['ref_question'].lower().split()

    bleu_scores.append(sentence_bleu([ref_tok], gen_tok, smoothing_function=smooth))
    rr = scorer.score(r['ref_question'], r['question'])
    rouge1_scores.append(rr['rouge1'].fmeasure)
    rougeL_scores.append(rr['rougeL'].fmeasure)
    meteor_scores.append(nltk_meteor([ref_tok], gen_tok))

print("=" * 55)
print("  QUESTION GENERATION METRICS (Validation Set)")
print("=" * 55)
print(f"  BLEU      : {np.mean(bleu_scores):.4f}")
print(f"  ROUGE-1   : {np.mean(rouge1_scores):.4f}")
print(f"  ROUGE-L   : {np.mean(rougeL_scores):.4f}")
print(f"  METEOR    : {np.mean(meteor_scores):.4f}")
print("=" * 55)


  QUESTION GENERATION METRICS (Validation Set)
  BLEU      : 0.0408
  ROUGE-1   : 0.2407
  ROUGE-L   : 0.2163
  METEOR    : 0.1433


In [14]:
# ─────────────────────────────────────────────────────────────────────────
# VALIDATION: Answer Extraction Quality
# ─────────────────────────────────────────────────────────────────────────
# Check how often the extracted answer shares words with the RACE reference answer.

overlap_30 = 0  # 30%+ word overlap
overlap_50 = 0  # 50%+ word overlap

for r in results:
    ext_tokens = set(clean_tokens(r['answer']))
    ref_tokens = set(clean_tokens(r['ref_answer']))
    if not ref_tokens:
        continue
    overlap_pct = len(ext_tokens & ref_tokens) / len(ref_tokens)
    if overlap_pct >= 0.3:
        overlap_30 += 1
    if overlap_pct >= 0.5:
        overlap_50 += 1

print("=" * 55)
print("  ANSWER EXTRACTION QUALITY (Validation Set)")
print("=" * 55)
print(f"  30%+ word overlap with reference: {overlap_30}/{len(results)} = {overlap_30/len(results):.1%}")
print(f"  50%+ word overlap with reference: {overlap_50}/{len(results)} = {overlap_50/len(results):.1%}")
print("=" * 55)


  ANSWER EXTRACTION QUALITY (Validation Set)
  30%+ word overlap with reference: 4729/8838 = 53.5%
  50%+ word overlap with reference: 3693/8838 = 41.8%


In [15]:
# ─────────────────────────────────────────────────────────────────────────
# VALIDATION: Distractor Ranker Quality
# ─────────────────────────────────────────────────────────────────────────
# For 500 samples, generate distractors and check:
# 1. Are they different from the correct answer? (should be yes)
# 2. Do they have reasonable length compared to the answer?

import re

def extract_candidates(article, correct_answer):
    sentences = sent_tokenize(str(article))
    candidates = []
    ans_lower = correct_answer.lower().strip()
    for sent in sentences:
        sent = sent.strip()
        if sent.lower().strip() == ans_lower or len(sent.split()) < 3:
            continue
        candidates.append(sent)
        parts = re.split(r'[,;]', sent)
        for part in parts:
            part = part.strip()
            if len(part.split()) >= 3 and part.lower() != ans_lower:
                candidates.append(part)
    seen = set()
    return [c for c in candidates if c.lower() not in seen and not seen.add(c.lower())]

N_EVAL = 500
not_answer = 0
reasonable_len = 0
total_dist = 0

for r in results[:N_EVAL]:
    candidates = extract_candidates(r['source'], r['ref_answer'])
    if not candidates:
        continue

    scored = []
    for cand in candidates:
        feats = distractor_features(cand, r['ref_answer'], r['source'], tfidf)
        prob = rf_distractor.predict_proba([feats])[0][1]
        scored.append((cand, prob))

    scored.sort(key=lambda x: -x[1])
    top3 = [s[0] for s in scored[:3]]

    for d in top3:
        total_dist += 1
        if d.lower().strip() != r['ref_answer'].lower().strip():
            not_answer += 1
        d_len = len(d.split())
        a_len = len(r['ref_answer'].split())
        if 0.3 <= d_len / max(1, a_len) <= 3.0:
            reasonable_len += 1

print("=" * 55)
print("  DISTRACTOR QUALITY (Validation Set, 500 samples)")
print("=" * 55)
print(f"  Not identical to answer: {not_answer}/{total_dist} = {not_answer/max(1,total_dist):.1%}")
print(f"  Reasonable length:       {reasonable_len}/{total_dist} = {reasonable_len/max(1,total_dist):.1%}")
print("=" * 55)


  DISTRACTOR QUALITY (Validation Set, 500 samples)
  Not identical to answer: 842/842 = 100.0%
  Reasonable length:       560/842 = 66.5%


In [16]:
# ─────────────────────────────────────────────────────────────────────────
# VALIDATION: Sample Outputs (10 examples)
# ─────────────────────────────────────────────────────────────────────────
print("── Sample Pipeline Outputs ──")
for i in range(min(10, len(results))):
    r = results[i]
    print(f"\n{'='*60}")
    print(f"[{i+1}]")
    print(f"  Source:        {r['source'][:80]}...")
    print(f"  Q-Type:        {r['q_type']}")
    print(f"  Generated Q:   {r['question']}")
    print(f"  Reference Q:   {r['ref_question']}")
    print(f"  Extracted Ans: {r['answer'][:80]}...")
    print(f"  Reference Ans: {r['ref_answer'][:80]}...")


── Sample Pipeline Outputs ──

[1]
  Source:        But aunt Fanny asked my mother not to tell the recipe  of making the strawberry ...
  Q-Type:        what
  Generated Q:   What aunt fanny asked my mother not to tell the recipe of making?
  Reference Q:   Who owns the recipe of making the best strawberry jam?
  Extracted Ans: But aunt Fanny asked my mother not to tell the recipe  of making the strawberry ...
  Reference Ans: The writer's mother....

[2]
  Source:        Maybe it was the recipe that brought success to aunt Fanny....
  Q-Type:        what
  Generated Q:   What maybe it was the recipe that brought success to aunt fanny?
  Reference Q:   Why do people in the town think aunt Fanny is poor?
  Extracted Ans: Maybe it was the recipe that brought success to aunt Fanny....
  Reference Ans: Because she had no close friends to shared her success....

[3]
  Source:        The list below shows the benefits of some colors in fruits and vegetables....
  Q-Type:        how
  Generate

In [17]:
# ─────────────────────────────────────────────────────────────────────────
# SUMMARY: All saved models
# ─────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("  ALL MODELS SAVED SUCCESSFULLY")
print("=" * 60)
for f in sorted(MODELS_DIR.glob("*.pkl")):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:35s}  {size_mb:.1f} MB")
print("=" * 60)
print()
print("These models will be loaded by the Streamlit app.")
print("Next step: build pipeline/module_a.py and pipeline/module_b.py")


  ALL MODELS SAVED SUCCESSFULLY
  count_vectorizer.pkl                 0.2 MB
  lr_answer_verifier.pkl               0.0 MB
  lr_sentence_selector.pkl             0.0 MB
  nb_question_type.pkl                 0.9 MB
  rf_distractor_ranker.pkl             18.6 MB
  tfidf_vectorizer.pkl                 0.6 MB

These models will be loaded by the Streamlit app.
Next step: build pipeline/module_a.py and pipeline/module_b.py
